# DSE Baselines — Notebook 2: PyG
**Methods**: DSGAT, A3Net, HSTrans using **UNIFIED SPLITS**

- **Split A**: Warm-start (pair-wise) — all 3 methods
- **Split B**: Drug cold-start — all 3 methods

DSGAT/A3Net use **mask matrices** (generated from unified splits).
HSTrans uses **pair-based** splits with imported functions.

In [ ]:
%%time
REPO_URL = 'https://github.com/YOUR_USERNAME/DSE.git'
!git clone $REPO_URL /content/DSE
%cd /content/DSE

!pip install -q torch torch-geometric rdkit-pypi scipy pandas scikit-learn

import os
for d in ['DSGAT_A', 'DSGAT_B', 'A3Net_A', 'A3Net_B', 'HSTrans_A', 'HSTrans_B']:
    os.makedirs(f'results/{d}', exist_ok=True)

In [ ]:
# Generate unified mask matrices from splits
import sys, pickle, numpy as np
sys.path.insert(0, '/content/DSE/shared_data')
from split_adapter import save_mask_mat, load_splits

with open('/content/DSE/shared_data/drug_side.pkl', 'rb') as f:
    freq = np.array(pickle.load(f))

print(f'Freq matrix: {freq.shape}')
print(f'Positive pairs: {(freq > 0).sum()}')

---
## Split A: Warm-start
### 1. DSGAT

In [ ]:
%%time
# Generate warm-start masks for DSGAT
save_mask_mat(freq, 'A', 10, '/content/DSE/DSGAT/data_WS/mask_mat_750.mat')

%cd /content/DSE/DSGAT
!mkdir -p result_WS data_WS/processed

import glob, os, numpy as np, pandas as pd
for f in glob.glob('/content/DSE/DSGAT/data_WS/processed/*'):
    try: os.remove(f)
    except: pass

# DSGAT already reads mask_mat_750.mat in split_data()
# So replacing the mask file = using our unified splits
%run WS_v4.py --tenfold --epoch 200 --wd 0.001 --cuda_name cuda:0

result_dirs = glob.glob('/content/DSE/DSGAT/result_WS/*')
if result_dirs:
    result_folder = sorted(result_dirs)[-1]
    pred_mat = pd.read_csv(f'{result_folder}/pred_result.csv', header=None).values
    os.makedirs('/content/DSE/results/DSGAT_A', exist_ok=True)
    for fold in range(10):
        _, test_data = load_splits(fold, 'A')
        d_idx = test_data[:, 0].astype(int)
        s_idx = test_data[:, 1].astype(int)
        np.save(f'/content/DSE/results/DSGAT_A/fold_{fold}_labels.npy', freq[d_idx, s_idx])
        np.save(f'/content/DSE/results/DSGAT_A/fold_{fold}_preds.npy', pred_mat[d_idx, s_idx])

### 2. A3Net

In [ ]:
%%time
save_mask_mat(freq, 'A', 10, '/content/DSE/A-3Net-master/data/mask_mat_750.mat')

# Copy real data files (override possible Git LFS pointers)
!cp -f /content/DSE/shared_data/raw_frequency_750.mat /content/DSE/A-3Net-master/data/
!cp -f /content/DSE/shared_data/drug_SMILES_750.csv /content/DSE/A-3Net-master/data/
!cp -f /content/DSE/shared_data/side_effect_label_750.mat /content/DSE/A-3Net-master/data/

%cd /content/DSE/A-3Net-master
!mkdir -p result_WS warm-scence_data data_WS/processed

import glob, os, numpy as np, pandas as pd
for f in glob.glob('/content/DSE/A-3Net-master/data_WS/processed/*'):
    try: os.remove(f)
    except: pass

%run warm-scence.py --tenfold --epoch 200 --cuda_name cuda:0

result_dirs = glob.glob('/content/DSE/A-3Net-master/result_WS/*')
if result_dirs:
    result_folder = sorted(result_dirs)[-1]
    pred_mat = pd.read_csv(f'{result_folder}/pred_result.csv', header=None).values
    os.makedirs('/content/DSE/results/A3Net_A', exist_ok=True)
    for fold in range(10):
        _, test_data = load_splits(fold, 'A')
        d_idx = test_data[:, 0].astype(int)
        s_idx = test_data[:, 1].astype(int)
        np.save(f'/content/DSE/results/A3Net_A/fold_{fold}_labels.npy', freq[d_idx, s_idx])
        np.save(f'/content/DSE/results/A3Net_A/fold_{fold}_preds.npy', pred_mat[d_idx, s_idx])

### 3. HSTrans
HSTrans uses pair-based splits. Need to:
1. Run `identify_sub()` to generate substructure files
2. Include negative samples in train/test (HSTrans expects 1:1 pos:neg)
3. Patch device to cuda:0

In [ ]:
%%time
import subprocess, random, sys, os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

%cd /content/DSE/HSTrans/HSTrans_original/HSTrans
os.makedirs('predictResult', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('data/sub', exist_ok=True)

# Patch device
subprocess.run(['sed', '-i', "s/self.device = 'cpu'/self.device = 'cuda:0'/g", 'Net.py'])

sys.path.insert(0, '/content/DSE/HSTrans/HSTrans_original/HSTrans')
import main as main_module
from main import (load_drug_smile, identify_sub, main as hstrans_main)
from Net import Trans
from smiles2vector import drug2emb_encoder

# 1. Monkeypatch predict to save all labels (not just non-zero)
def custom_predict(model, device, test_loader):
    total_preds = torch.Tensor()
    total_labels = torch.Tensor()
    model.eval()
    with torch.no_grad():
        for batch_idx, (Drug, SE, DrugMask, SEMsak, Label) in enumerate(test_loader):
            DrugMask = DrugMask.to(device)
            SEMsak = SEMsak.to(device)
            Label = torch.FloatTensor([int(item) for item in Label])
            out, _, _ = model(Drug, SE, DrugMask, SEMsak)
            total_preds = torch.cat((total_preds, out.cpu()), 0)
            total_labels = torch.cat((total_labels, Label.cpu()), 0)
    return total_labels.numpy().flatten(), total_preds.numpy().flatten()

main_module.predict = custom_predict

# 2. Fast Data Encoder (Loads sub files with correct suffix _0)
class FastData_Encoder(Dataset):
    def __init__(self, list_IDs, labels, df_dti, k):
        self.labels = labels
        self.list_IDs = list_IDs
        self.df = df_dti
        self.k = k
        self.SE_index = np.load("data/sub/SE_sub_index_50_0.npy").astype(int)
        self.SE_mask = np.load("data/sub/SE_sub_mask_50_0.npy")

    def __len__(self):
        return len(self.list_IDs)

    def __getitem__(self, idx_pos):
        index = self.list_IDs[idx_pos]
        d = self.df.iloc[index]['Drug_smile']
        s = int(self.df.iloc[index]['SE_id'])

        d_v, input_mask_d = drug2emb_encoder(d)
        s_v = self.SE_index[s, :]
        input_mask_s = self.SE_mask[s, :]
        y = self.labels[idx_pos]
        return d_v, s_v, input_mask_d, input_mask_s, y


drug_dict, drug_smile = load_drug_smile('/content/DSE/shared_data/drug_SMILES_750.csv')

# Build all data for substructure identification
pos_indices = np.argwhere(freq > 0)
all_data_for_sub = [(int(s), drug_smile[int(d)], freq[d, s]) for d, s in pos_indices]
identify_sub(all_data_for_sub, 0)  # Generate SE substructure files

# Sample negatives from freq matrix
neg_pairs = np.argwhere(freq == 0)

os.makedirs('/content/DSE/results/HSTrans_A', exist_ok=True)

for fold in range(10):
    print(f'\n===== HSTrans Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'A')
    
    # Positive pairs: (SE_id, Drug_smile_str, freq)
    data_train_pos = [(int(r[1]), drug_smile[int(r[0])], r[2]) for r in train_pos]
    data_test_pos = [(int(r[1]), drug_smile[int(r[0])], r[2]) for r in test_pos]
    
    # Sample negative pairs (1:1 ratio)
    rng = random.Random(42 + fold)
    neg_sample_train = rng.sample(range(len(neg_pairs)), len(train_pos))
    neg_sample_test = rng.sample(range(len(neg_pairs)), len(test_pos))
    
    data_train_neg = [(int(neg_pairs[i][1]), drug_smile[int(neg_pairs[i][0])], 0) for i in neg_sample_train]
    data_test_neg = [(int(neg_pairs[i][1]), drug_smile[int(neg_pairs[i][0])], 0) for i in neg_sample_test]
    
    data_train = data_train_pos + data_train_neg
    data_test = data_test_pos + data_test_neg
    
    df_train = pd.DataFrame(data_train, columns=['SE_id', 'Drug_smile', 'Label'])
    df_test = pd.DataFrame(data_test, columns=['SE_id', 'Drug_smile', 'Label'])
    
    training_set = FastData_Encoder(df_train.index.values, df_train.Label.values, df_train, fold)
    testing_set = FastData_Encoder(df_test.index.values, df_test.Label.values, df_test, fold)
    
    training_gen = DataLoader(training_set, batch_size=128, shuffle=True)
    testing_gen = DataLoader(testing_set, batch_size=128, shuffle=False)
    
    # main() saves to predictResult/total_labels_{k}.npy, total_preds_{k}.npy
    hstrans_main(training_gen, testing_gen, Trans, 1e-4, 200, 0.01, 40, 'cuda:0', True, fold)
    
    # Copy predictions to unified results (Note: HSTrans regression metrics come from original predict, binary metrics need manual eval)
    labels = np.load(f'predictResult/total_labels_{fold}.npy')
    preds = np.load(f'predictResult/total_preds_{fold}.npy')
    np.save(f'/content/DSE/results/HSTrans_A/fold_{fold}_labels.npy', labels)
    np.save(f'/content/DSE/results/HSTrans_A/fold_{fold}_preds.npy', preds)


---
## Split B: Drug Cold-start
Only DSGAT, A3Net, HSTrans (drug features are from SMILES graph, not freq matrix)

In [ ]:
%%time
# DSGAT Split B
save_mask_mat(freq, 'B', 10, '/content/DSE/DSGAT/data_WS/mask_mat_750.mat')

# Clean previous processed data
import glob, os, numpy as np, pandas as pd
for f in glob.glob('/content/DSE/DSGAT/data_WS/processed/*'):
    try: os.remove(f)
    except: pass

%cd /content/DSE/DSGAT
%run WS_v4.py --tenfold --epoch 200 --wd 0.001 --cuda_name cuda:0

result_dirs = glob.glob('/content/DSE/DSGAT/result_WS/*')
if result_dirs:
    result_folder = sorted(result_dirs)[-1]
    pred_mat = pd.read_csv(f'{result_folder}/pred_result.csv', header=None).values
    os.makedirs('/content/DSE/results/DSGAT_B', exist_ok=True)
    for fold in range(10):
        _, test_data = load_splits(fold, 'B')
        d_idx = test_data[:, 0].astype(int)
        s_idx = test_data[:, 1].astype(int)
        np.save(f'/content/DSE/results/DSGAT_B/fold_{fold}_labels.npy', freq[d_idx, s_idx])
        np.save(f'/content/DSE/results/DSGAT_B/fold_{fold}_preds.npy', pred_mat[d_idx, s_idx])

In [ ]:
%%time
# A3Net Split B
save_mask_mat(freq, 'B', 10, '/content/DSE/A-3Net-master/data/mask_mat_750.mat')

import glob, os, numpy as np, pandas as pd
for f in glob.glob('/content/DSE/A-3Net-master/data_WS/processed/*'):
    try: os.remove(f)
    except: pass

%cd /content/DSE/A-3Net-master
%run warm-scence.py --tenfold --epoch 200 --cuda_name cuda:0

result_dirs = glob.glob('/content/DSE/A-3Net-master/result_WS/*')
if result_dirs:
    result_folder = sorted(result_dirs)[-1]
    pred_mat = pd.read_csv(f'{result_folder}/pred_result.csv', header=None).values
    os.makedirs('/content/DSE/results/A3Net_B', exist_ok=True)
    for fold in range(10):
        _, test_data = load_splits(fold, 'B')
        d_idx = test_data[:, 0].astype(int)
        s_idx = test_data[:, 1].astype(int)
        np.save(f'/content/DSE/results/A3Net_B/fold_{fold}_labels.npy', freq[d_idx, s_idx])
        np.save(f'/content/DSE/results/A3Net_B/fold_{fold}_preds.npy', pred_mat[d_idx, s_idx])

In [ ]:
%%time
# HSTrans Split B
%cd /content/DSE/HSTrans/HSTrans_original/HSTrans
from split_adapter import load_test_drugs
import random, os
import pandas as pd, numpy as np
from torch.utils.data import DataLoader
from Net import Trans
from main import main as hstrans_main

os.makedirs('/content/DSE/results/HSTrans_B', exist_ok=True)

for fold in range(10):
    print(f'\n===== HSTrans Cold-start Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    
    data_train_pos = [(int(r[1]), drug_smile[int(r[0])], r[2]) for r in train_pos]
    data_test_pos = [(int(r[1]), drug_smile[int(r[0])], r[2]) for r in test_pos]
    
    rng = random.Random(42 + fold)
    neg_sample_train = rng.sample(range(len(neg_pairs)), min(len(train_pos), len(neg_pairs)))
    neg_sample_test = rng.sample(range(len(neg_pairs)), min(len(test_pos), len(neg_pairs)//10))
    
    data_train_neg = [(int(neg_pairs[i][1]), drug_smile[int(neg_pairs[i][0])], 0) for i in neg_sample_train]
    data_test_neg = [(int(neg_pairs[i][1]), drug_smile[int(neg_pairs[i][0])], 0) for i in neg_sample_test]
    
    data_train = data_train_pos + data_train_neg
    data_test = data_test_pos + data_test_neg
    
    df_train = pd.DataFrame(data_train, columns=['SE_id', 'Drug_smile', 'Label'])
    df_test = pd.DataFrame(data_test, columns=['SE_id', 'Drug_smile', 'Label'])
    
    training_set = FastData_Encoder(df_train.index.values, df_train.Label.values, df_train, fold)
    testing_set = FastData_Encoder(df_test.index.values, df_test.Label.values, df_test, fold)
    
    training_gen = DataLoader(training_set, batch_size=128, shuffle=True)
    testing_gen = DataLoader(testing_set, batch_size=128, shuffle=False)
    
    hstrans_main(training_gen, testing_gen, Trans, 1e-4, 200, 0.01, 40, 'cuda:0', True, fold)
    
    labels = np.load(f'predictResult/total_labels_{fold}.npy')
    preds = np.load(f'predictResult/total_preds_{fold}.npy')
    np.save(f'/content/DSE/results/HSTrans_B/fold_{fold}_labels.npy', labels)
    np.save(f'/content/DSE/results/HSTrans_B/fold_{fold}_preds.npy', preds)

---
## Unified Evaluation

In [ ]:
%cd /content/DSE
!python shared_data/unified_eval.py --results_dir ./results